# Part 1: Build a deep research agent

In this part, you will implement a deep research agent in LangGraph and analyze the requests it sends to a language model. The agent divides a question into subtopics, researches them in parallel, and writes a report with citations to source passages.

In Assignment 3, each simulated request represented a single LLM call. In an agentic workload, later LLM calls may wait for earlier calls or tools to finish, and related calls often share long prompt prefixes. You will use the recorded traces to study these execution dependencies, prefix reuse, and the effect of concurrency. Parts 2 and 3 examine how to schedule and cache this workload.

Complete the following:

- `research_app/concurrency.py`: implement `acquire`, `release`, and `set_limit` on `AdjustableCap`.
- `research_app/agent.py`: implement the graph nodes, routing functions, and `build_graph`.
- `research_app/analysis.py`: implement `critical_path`, `shared_prefix_share`, `reuse_gaps`, and `kv_footprint`, which section 5 uses to analyze your runs.

See the [Part 1 handout](../docs/part1_handout.md) for the full specification.

| Section | Task | Estimated time |
|---|---|---|
| 1. Corpus and tools | Inspect the corpus, search results, and questions | 15 min |
| 2. Concurrency cap | Implement `AdjustableCap` | 1 h |
| 3. Research agent | Implement the graph | 3–4 h |
| 4. Run the agent | Three runs: C = 2, C = 8, and C = 8 with tool failures | 30 min |
| 5. Workload analysis | Implement the analysis functions; replay your trace on a simulated GPU | 3 h |
| Questions | Answer Q1–Q4 | 2–3 h |

**Estimated time:** 10–12 hours.

**Grading (20 points):** implementation correctness (12 points: cap 2, graph 6, analysis 4); answers to Q1–Q4 (8 points).

## Before you begin

Complete the [environment setup](../README.md#setup) before running this notebook. Select **CS 378 Assignment 4** as the notebook kernel so its cells use the `.venv` environment where you installed the assignment dependencies. (When you open a new terminal, run `source .venv/bin/activate` from the repository root; activating `.venv` in a terminal does not change the kernel of an already-open notebook.)

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())
plt.rcParams["figure.dpi"] = 100

BACKEND = "bedrock" if os.environ.get("AGENTKV_BEDROCK_MODEL") else "fake"
print(f"Working in {Path.cwd()}")
hint = "" if BACKEND == "bedrock" else "  (set AGENTKV_BEDROCK_MODEL to use Bedrock)"
print(f"LLM backend: {BACKEND}{hint}")

## 1. Corpus and tools

Before implementing the agent, you will explore the offline corpus of 48 Wikipedia articles on six systems topics (each saved at two revisions, `v1` and `v2`). Rather than querying the live web, we supply the following two tools over this corpus:

- `search(query, k=5)` ranks articles by BM25 relevance and returns up to `k` matching `SearchHit` results, each containing the `doc_id`, title, relevance score, snapshot hash, and a short snippet.
- `fetch(doc_id)` retrieves the full article text and revision metadata (`Doc`) for a given `doc_id`.

When called through the agent's tool wrapper, both tools also simulate reproducible delays and occasional transient failures using a seeded latency model so you can examine how tool calls affect the agent's progress.

Use this local corpus while implementing the graph and running the controlled experiments in sections 4 and 5. After completing the required work, you can also connect your agent to live web search in section 6.

In [ ]:
from research_app.corpus import Corpus
from research_app.questions import load_questions

corpus = Corpus("v2")
print({topic: len(ids) for topic, ids in corpus.topics().items()})

for hit in corpus.search("how does raft elect a leader", k=3):
    print(f"{hit.doc_id:40} score {hit.score:.2f}  {hit.title}")

In [ ]:
doc = corpus.fetch(corpus.search("how does raft elect a leader", k=1)[0].doc_id)
print(doc.title, f"({len(doc.text)} characters, snapshot sha256 {doc.sha256[:12]})")
print(doc.text[:600], "...")

The next cell lists the first five of the 16 public research questions. Try passing a few queries to `corpus.search()` and inspect the results; if a query does not retrieve the article you expected, try rephrasing it to see how BM25 ranking responds.

In [ ]:
questions = load_questions(public=True)
for q in questions[:5]:
    print(f"{q['id']:18} {q['question']}")
print(f"... {len(questions)} questions")

## 2. Concurrency cap

Across all active reports, the application enforces a shared limit **C** on how many LLM calls can run at once (in Parts 2 and 3, serving policies will control the corresponding limit via `target_concurrency`).

Complete the following methods on `AdjustableCap` in `research_app/concurrency.py`:

- `acquire()`: Wait until `in_flight < limit`, reserve a slot, and update the counters for current in-flight calls, total grants, and peak concurrency.
- `release()`: Return a slot and wake waiting calls so they can re-check whether a slot is available.
- `set_limit(n)`: Update the limit and wake waiting calls. If the limit is lowered below `in_flight`, let running calls finish without interruption while keeping new calls waiting until `in_flight < limit`.

The supplied `slot()` context manager calls `acquire()` and `release()` around an entire LLM operation (including retries, so a retry keeps its existing slot). Use an `asyncio.Condition` to coordinate waiting coroutines, then run the unit test cell below.

In [ ]:
check("tests/test_part1_concurrency.py")

The next cell tests how your limiter responds when the cap changes mid-run. Six calls start with an initial limit of 3, and while the first three are running, the limit drops to 1:

- **Upper plot (per-call timeline):** Yellow bars show time waiting for a slot; blue bars show time running.
- **Lower plot (concurrency over time):** Shows the number of running calls alongside the active limit (dashed line).

After the limit drops to 1, the three running calls should finish normally (so `in_flight` temporarily exceeds the new limit without counting as an error), and the remaining three calls should then run one at a time.

In [ ]:
from research_app.concurrency import AdjustableCap
from research_app.visualize import run_cap_demo, plot_cap_demo

cap = AdjustableCap(3)
demo = await run_cap_demo(cap)
plot_cap_demo(demo)
plt.show()
print(f"Peak running calls: {demo.peak} (expected 3)")
print(f"Calls incorrectly started at or above the limit: {demo.over_limit} (expected 0)")

## 3. Research agent

Next, implement the research agent in `research_app/agent.py`. The workflow below shows how the agent plans subtopics, researches them in parallel, checks subtopic coverage (running at most one follow-up branch if needed), synthesizes a cited report, and validates citations (with at most one repair call):

![Research graph: plan, parallel research branches, coverage check, optional follow-up, synthesize, validate, and optional repair.](../docs/figures/research_workflow.svg)

See the [Part 1 handout](../docs/part1_handout.md) for a quick overview of the LangGraph primitives (`StateGraph`, edges, conditional edges, `Send` fan-out, and join) and the required `op_id` and `order` conventions. Keep three rules in mind as you implement the nodes:

- **Route all calls through `ctx.client`:** The tracing client records the trace and enforces per-report budgets (at most 8 LLM calls, 5 searches, 15 fetches, and 3 attempts per operation).
- **Record exact `deps` and use `research_app/prompts.py`:** Each operation's `deps` list defines the edges of the exported DAG (an `extract` call depends on all fetches in its branch, including failed ones, though only fetched documents enter its prompt). The prompt builders place shared instructions and documents first so calls share prefixes.
- **Handle permanent tool failures gracefully:** A call that fails permanently returns `None`; return a typed `BranchResult` rather than raising an exception so the rest of the report can proceed.

Start by implementing `build_graph`, then fill in the nodes and routing functions. The cell below runs a single report through `ResearchAgent.research(topic)` and stops with `NotImplementedError` at the first unfinished node.

In [ ]:
from IPython.display import Markdown, display
from research_app import ResearchAgent
from research_app.llm import make_llm
from research_app.search import LocalSearch

agent = ResearchAgent(llm=make_llm(BACKEND), search=LocalSearch(), cap=2)
result = await agent.research(questions[0]["question"])
print(f"Citation checks passed: {result.valid}; LLM calls: {result.recording.counts()['llm_calls']}")
print("Errors:", result.recording.errors)

In [ ]:
display(Markdown(result.markdown()))

In [ ]:
check("tests/test_part1_agent.py", "tests/test_part1_citations.py", "tests/test_part1_llm.py",
      "tests/test_part1_trace_export.py")

## 4. Run the agent

Run all 16 public questions three times, launching one report every 2 seconds:

| Run | Directory | Purpose |
|---|---|---|
| C = 2 | `runs/c2/` | LLM calls queue for slots |
| C = 8 | `runs/c8/` | Little queueing |
| C = 8, 20% of tool attempts fail | `runs/c8-faults/` | Retries, follow-up branches, and repairs |

The runs use the backend selected in the setup cell (`bedrock` if `AGENTKV_BEDROCK_MODEL` is set, otherwise `fake`); report which one you used. With `FakeLLM` each run takes one to two minutes; Bedrock runs take a few minutes.

Each run writes `trace.jsonl` (every LLM and tool event, with timestamps and prompt `prefix_ids`), `reports.json`, `summary.json`, and `replay.json` (the run as a trace that the Vidur simulator can replay). Check that every run reports `16/16 valid` before moving on.

In [ ]:
!{sys.executable} -m research_app.run --backend {BACKEND} --cap 2 --replay --out runs/c2
!{sys.executable} -m research_app.run --backend {BACKEND} --cap 8 --replay --out runs/c8
FAULTS = "--failure-rate 0.2"
!{sys.executable} -m research_app.run --backend {BACKEND} --cap 8 {FAULTS} --replay --out runs/c8-faults

## 5. Workload analysis

Implement the four functions in `research_app/analysis.py`, then run the tests. Each takes the operations of a recorded run (`load_run("runs/c2")`); their docstrings and the handout specify the results exactly.

In [ ]:
check("tests/test_part1_analysis.py")

### 5a. Critical path

`critical_path(run, report)` walks back from a report's last operation through the dependency that finished last, and splits each step into the gap before it started, its wait for an LLM slot, and its service time. Together they add up to the report's latency.

The DAG below highlights the path of one report at C = 2. Each box shows the operation's service time and, for LLM calls, its wait for a slot. The bar chart compares every report's path at the three settings.

In [ ]:
from research_app.visualize import plot_critical_paths, plot_research_dag

fig, steps = plot_research_dag("runs/c2", report_id="R04")   # choose any report
plt.show()
fig, totals = plot_critical_paths(["runs/c2", "runs/c8", "runs/c8-faults"])
plt.show()
for row in totals:
    print(f"{row['run']}: mean per report: slot wait {row['wait']:.1f} s, LLM {row['llm']:.1f} s, "
          f"tools {row['tool']:.1f} s")

### 5b. Prefix reuse

Each completed LLM call in `trace.jsonl` records `prefix_ids` for its full 256-token prompt blocks; two calls share an id only if their prompts are identical through that block. `shared_prefix_share` measures how much prompt content appeared in earlier calls; `reuse_gaps` measures how long each reused block sat unused before its reuse, which is how long a cache would have had to keep it.

In [ ]:
from research_app.analysis import load_run, shared_prefix_share
from research_app.visualize import plot_prefix_reuse, plot_reuse_gaps

for run_dir in ("runs/c2", "runs/c8"):
    calls = load_run(run_dir).llm_calls()
    print(f"{run_dir}: {len(calls)} LLM calls; shared prefix share {shared_prefix_share(calls):.1%}")
    plot_prefix_reuse(calls, shared_prefix_share)
    plt.show()
plot_reuse_gaps(["runs/c2", "runs/c8"])
plt.show()

### 5c. Concurrency and memory

The first plots compare the runs: mean report latency, makespan, and slot waits, then each report's timeline (yellow: an LLM call waiting for a slot; blue: running; gray: tools; black tick: arrival).

A hosted API hides its GPUs, so these runs cannot show memory pressure. `kv_footprint` measures the KV cache that the running calls would need on one Llama-3-8B GPU (128 KiB per token in bf16), with and without storing identical prefix blocks once.

In [ ]:
from research_app.visualize import plot_kv_footprint, plot_run_comparison, plot_run_timelines

fig, measurements = plot_run_comparison(["runs/c2", "runs/c8"])
plt.show()
plot_run_timelines(["runs/c2", "runs/c8"])
plt.show()
fig, peaks = plot_kv_footprint(["runs/c2", "runs/c8"])
plt.show()
for row, peak in zip(measurements, peaks):
    print(f"C = {row['cap']}: {row['valid']}/{row['reports']} valid; "
          f"peak {row['llm_peak']} concurrent LLM calls; "
          f"mean slot wait {row['mean_slot_wait']:.2f} s; peak KV {peak['private_gb']:.1f} GB "
          f"({peak['shared_gb']:.1f} GB sharing prefixes)")

Now replay your C = 8 run on a simulated A100 (the Vidur simulator of Parts 2 and 3), varying the concurrency cap and the KV-cache capacity. `viz.record` replays `replay.json` with the same dependencies, prompts, outputs, and tool delays; `kv_tokens` sets the cache size. The policy is `policies/fifo.py`, a complete FIFO-eviction policy whose `cap` parameter sets the concurrency limit. Each replay takes about a second.

In [ ]:
caps = (1, 2, 4, 8, 16)
kv_sizes = (12_288, 16_384, 49_152, 393_216)       # tokens; 128 KiB each
latency = {}
for kv in kv_sizes:
    for cap in caps:
        rec = viz.record("runs/c8/replay.json", "policies/fifo.py", params={"cap": cap}, kv_tokens=kv)
        latency[kv, cap] = sum(rec.flows.values()) / len(rec.flows)

fig, ax = plt.subplots(figsize=(7, 3.6))
for kv in kv_sizes:
    label = f"{kv:,} tokens ({kv * 131072 / 1e9:.1f} GB)"
    ax.plot(caps, [latency[kv, c] for c in caps], marker="o", label=label)
ax.set(xscale="log", xticks=caps, xticklabels=caps, xlabel="Concurrency cap",
       ylabel="Mean report latency (s)")
ax.legend(title="KV-cache capacity", frameon=False)
plt.show()

### 5d. Failures

In `runs/c8-faults`, tool calls fail transiently 20% of the time. The tracing client retries each operation up to three times with backoff; a branch whose fetches all fail gets a follow-up branch, and a report whose citations fail validation gets one repair call. Compare the failure run with `runs/c8`.

In [ ]:
from research_app.analysis import critical_path

for run_dir in ("runs/c8", "runs/c8-faults"):
    run = load_run(run_dir)
    ops = run.ops.values()
    latencies = sorted(max(o.end for o in run.report_ops(r)) - run.arrivals[r] for r in run.arrivals)
    on_path = {s.op_id for r in run.arrivals for s in critical_path(run, r)}
    retried = [o for o in ops if o.attempts > 1]
    print(f"{run_dir}: median latency {latencies[len(latencies) // 2]:.1f} s, max {latencies[-1]:.1f} s; "
          f"{len(retried)} retried operations "
          f"({sum(o.op_id in on_path for o in retried)} on a critical path), "
          f"{sum('/followup/' in o.op_id for o in ops if o.kind == 'llm')} follow-ups, "
          f"{sum(o.op_id.endswith('/repair') for o in ops)} repairs")

## Questions

Answer Q1–Q4 below, keeping each question above its answer. Include your answers, plots, and tables in `assignment4_report.pdf`, and include `runs/c2/`, `runs/c8/`, and `runs/c8-faults/` in your Canvas archive (see [Submission](../README.md#submission)).

**Q1. Critical path.** For one report at C = 2, show its DAG and critical path. Which operations determine when it finishes, and how much of its latency is slot waiting, LLM service, and tool service? How does the breakdown change at C = 8, across all reports? Which single change to the agent or the cap would shorten the typical C = 8 critical path most, and by roughly how much?

**Your answer:**

**Q2. Prefix reuse and retention.** What fraction of prompt tokens are in previously seen blocks, overall and by stage, and what content does each stage reuse? Using the reuse gaps, how long must a cache keep each kind of prefix for its reuse to hit, and which prefixes would you never keep? Use both runs.

**Your answer:**

**Q3. Concurrency and memory.** Compare C = 2 and C = 8 in a table: mean report latency, makespan, mean slot wait, peak concurrent calls, and peak KV footprint with and without prefix sharing. Then use your replay sweep: at which KV capacity does a higher cap stop helping, and why? Assuming Llama-3-8B in bf16 on one 80 GB A100 (about 16 GB of weights), which cap would you choose for this workload, and how much memory headroom would remain?

**Your answer:**

**Q4. Failures.** In the failure run, where do retries, follow-ups, and repairs appear, and which of them land on critical paths? How do they change the median and worst report latency? An LLM call keeps its slot while it backs off between attempts; at C = 2, would releasing the slot during backoff help or hurt mean report latency, and why?

**Your answer:**

## 6. [Optional] Research a topic on the web

Once you have finished the required implementation and analysis above, you can use your `ResearchAgent` to investigate any topic on the live web. This extension is optional and does not affect your grade.

Keep the same `ResearchAgent` with `BedrockLLM()` and replace `LocalSearch()` with one of the web-search adapters supplied in `research_app/search.py` (`"exa"`, `"tavily"`, `"perplexity"`, or `"openai"`). See the [Artificial Analysis Search API leaderboard](https://artificialanalysis.ai/agents/search-api) for a comparison of search providers, and the [Part 1 handout](../docs/part1_handout.md#optional-research-a-topic-on-the-web) for details on how each adapter retrieves and archives source text:

| Provider | Adapter and environment variable | Free access (checked October 30, 2026) |
|---|---|---|
| [Exa](https://exa.ai/pricing) | `ExaSearch`, `EXA_API_KEY` | $10 signup bonus plus $10/month; no card required. |
| [Tavily](https://www.tavily.com/pricing) | `TavilySearch`, `TAVILY_API_KEY` | 1,000 credits/month (or 4,000/month via [student offer](https://help.tavily.com/articles/6606514713-student-account)); no card required. |
| [Perplexity Search](https://docs.perplexity.ai/docs/getting-started/pricing) | `PerplexitySearch`, `PERPLEXITY_API_KEY` | Pay as you go. |
| [OpenAI web search](https://developers.openai.com/api/docs/guides/tools-web-search) | `OpenAISearch`, `OPENAI_API_KEY`, `AGENTKV_SEARCH_MODEL` | Pay as you go (also set `AGENTKV_SEARCH_MODEL`). |

Configure Bedrock ([Model access](../README.md#model-access)) and your search provider's API key in the environment before launching Jupyter, then set `RUN_WEB_RESEARCH = True` below.

In [ ]:
from datetime import datetime
from research_app import ResearchAgent
from research_app.llm import BedrockLLM
from research_app.search import make_search

RUN_WEB_RESEARCH = False
SEARCH_PROVIDER = "exa"  # or "tavily", "perplexity", "openai"
TOPIC = "What are the tradeoffs between passkeys and passwords for account recovery?"

if RUN_WEB_RESEARCH:
    web_agent = ResearchAgent(llm=BedrockLLM(), search=make_search(SEARCH_PROVIDER), cap=2)
    topic_run = Path("runs") / f"topic-{datetime.now():%Y%m%d-%H%M%S}"
    web_result = await web_agent.research(TOPIC, out=topic_run)
    print(f"Citation checks passed: {web_result.valid}; saved to {topic_run}")
    display(Markdown(web_result.markdown()))
else:
    print("Optional web research is disabled. Configure your providers, then set RUN_WEB_RESEARCH = True.")